# Byte-Pair Encoding
Algorithm initially developed to compress text and later adapted by OpenAI for tokenization. <br/>
Usef by GPT, GPT-2, RoBERTa, BAT, DeBERTa.

## Training
It's start after normalization and pre-tokenization. <br/>
It build vocubulary of all symbols in the training data. <br/>
Training data : "hug", "pug", "pun", "bun", "hugs" <br/>
Vocabulary : ["b", "g", "h", "n", "p", "s", "u"]

Note: GPT and GPT-2 use  byte-level BPE. It converts chars to bytes so it consumes less data.

Next step is to create subword from the vocabulart by merge rules <br />
We aplly this step multiple time until merge rules are satisfied. <br/>
<br />
In this case our words are repeating multiple time <br />
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)<br />
<br />

We split out words into chars <br />
("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)<br />
<br />

We can count that ug pair is repeating 20 times in all words. So we merge it into new token "ug" <br />
("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)<br />
<br />

So now we have two definitons: vocubulary and corpus. <br />
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug"] <br />
Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)<br />
<br />

And we continue like this until we reach the desired vocabulary size. <br />
Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un", "hug"] <br />
Corpus: ("hug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("hug" "s", 5)

## Tokenization algorithm

 - Normalization
 - Pre-tokenization
 - Splitting the words into individual characters
 - Applying the merge rules learned in order on those splits

## BPE

In [ ]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
# Load tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
tokenizer

We calculate each word with pretokizetion.

In [ ]:

from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)

And calculating base vocubulary for a corpus

In [ ]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(alphabet)

We also add special token for GPT2

In [ ]:
vocab = ["<|endoftext|>"] + alphabet.copy()

In [ ]:
vocab

Splitting words into characters for training BPE tokenizer

In [ ]:
splits = {word: [c for c in word] for word in word_freqs.keys()}
splits

In [ ]:
# Function that compute each word pair if it appears in the splits
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

In [ ]:
pair_freqs = compute_pair_freqs(splits)

In [ ]:
pair_freqs

In [ ]:
for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

In [ ]:
# Find the most frequent pair
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

Our first merge rule is ('Ġ', 't') -> 'Ġt', so we need to add it to our vocabulary.

In [ ]:
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")

In [ ]:
vocab

In [ ]:
splits

Now we need to apply merge in our splits

In [ ]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [ ]:
splits = merge_pair("Ġ", "t", splits)
print(splits["Ġtrained"])

Now we can loop until we reach the desired vocabulary size.

In [ ]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

In [ ]:
vocab

In [ ]:
print(merges)

Whole tokenization process below wrapped into single function.

In [ ]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

In [ ]:
tokenize("to be more tokenized in morning.")